<a href="https://colab.research.google.com/github/MuhammedYo/kmeans-vs-dbscan/blob/main/Data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 120.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.metrics.pairwise import cosine_similarity

# Helper function for text preprocessing (already defined, but included here for completeness within the function's context)
def preprocess_text(text):
    text = str(text).lower() # Ensure text is string and convert to lowercase
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove non-alphabetic characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Helper function for keyword extraction (already defined, but included here for completeness)
def get_top_keywords(texts, n_keywords=10):
    # Filter out empty strings before vectorizing
    texts = [text for text in texts if text and text.strip() != '']
    if not texts: # Handle case where all texts are empty after filtering
        return []
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    X = vectorizer.fit_transform(texts)
    sums = X.sum(axis=0)
    words = vectorizer.get_feature_names_out()
    keywords = [words[i] for i in np.argsort(sums.A1)[-n_keywords:][::-1]]
    return keywords

# Recommendation function (copied from cell 310d5fa1 for dependency)
def get_recommendations(article_id, df_clustered, embeddings, n_recommendations=5):
    # 1. Get the cluster label for the article_id
    # Assuming article_id is the index of the article in the original df/filtered_df
    if article_id not in df_clustered.index:
        # Convert article_id to int if it's a float, before checking index
        if pd.isna(article_id):
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])
        try:
            article_id = int(article_id)
        except ValueError:
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])

        if article_id not in df_clustered.index:
            # If ID is valid but not in filtered_df (e.g., removed during preprocessing)
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])

    target_cluster_label = df_clustered.loc[article_id, 'cluster']

    # FIX: Get the correct embedding index for the target article from the filtered_df
    target_embedding_index = df_clustered.index.get_loc(article_id)
    target_embedding = embeddings[target_embedding_index].reshape(1, -1)

    # 2. Identify all other articles that belong to the same cluster
    cluster_articles = df_clustered[df_clustered['cluster'] == target_cluster_label]

    # Exclude the target article itself from the list of candidates for recommendations
    cluster_articles = cluster_articles[cluster_articles.index != article_id]

    if cluster_articles.empty:
        return pd.DataFrame(columns=['Title', 'Description', 'Similarity'])

    # 3. Get embeddings for articles within the cluster
    # Need to map cluster_articles.index to original embeddings indices
    # The embeddings array corresponds to `filtered_df` indices (0 to len(filtered_df)-1)
    # The `article_id` passed to get_recommendations is from `df_clustered.index`
    # So, need to find the correct `embeddings` indices for `cluster_articles`
    cluster_article_original_indices = [df_clustered.index.get_loc(idx) for idx in cluster_articles.index]
    cluster_embeddings = embeddings[cluster_article_original_indices]

    # 4. Calculate cosine similarity
    similarities = cosine_similarity(target_embedding, cluster_embeddings).flatten()

    # 5. Create a DataFrame for sorting and selection
    recommendations_df = cluster_articles.copy()
    recommendations_df['Similarity'] = similarities

    # 6. Sort by similarity in descending order
    recommendations_df = recommendations_df.sort_values(by='Similarity', ascending=False)

    # 7. Return the top n_recommendations articles
    return recommendations_df.head(n_recommendations)[['Title', 'Description', 'cluster_name', 'Similarity']]

# Define the trend analysis function - now returns DataFrame
def perform_trend_analysis(df_clustered, date_column_name):
    if not date_column_name or date_column_name not in df_clustered.columns:
        return None

    # Ensure date column is datetime objects for proper plotting and grouping
    df_clustered_copy = df_clustered.copy() # Operate on a copy to avoid SettingWithCopyWarning
    df_clustered_copy[date_column_name] = pd.to_datetime(df_clustered_copy[date_column_name])

    # Group by date (daily) and cluster_name, then count articles
    # Using .dt.date to group by calendar date, ignoring time components
    trend_data = df_clustered_copy.groupby([df_clustered_copy[date_column_name].dt.date, 'cluster_name']).size().reset_index(name='count')
    trend_data = trend_data.rename(columns={date_column_name: 'Date'})

    if trend_data.empty:
        return pd.DataFrame(columns=['Date', 'cluster_name', 'count'])

    return trend_data

# Define the main clustering function for Gradio - updated to return plot figures
def cluster_news_articles(file_obj, article_id_rec=None, n_recommendations_rec=5, date_column_name=None):
    # 1. Load Data
    df = pd.read_csv(file_obj.name)
    if 'Description' not in df.columns:
        raise ValueError("CSV file must contain a 'Description' column.")

    # 1.1 Process date column if provided
    initial_date_column_name = date_column_name # Store original name
    if date_column_name and date_column_name in df.columns:
        df[date_column_name] = pd.to_datetime(df[date_column_name], errors='coerce')
        df.dropna(subset=[date_column_name], inplace=True)
    else:
        # Set date_column_name to None if not valid or provided, to skip trend analysis later
        date_column_name = None

    texts = df['Description'].astype(str)

    # 2. Preprocess Text
    texts_clean = texts.apply(preprocess_text).tolist()
    # Filter out empty strings from texts_clean to avoid issues with SentenceTransformer
    original_indices = [i for i, text in enumerate(texts_clean) if text and text.strip() != '']
    filtered_texts_clean = [texts_clean[i] for i in original_indices]

    # Construct columns to retain in filtered_df
    cols_to_retain = ['Title', 'Description'] # Always retain Title and Description
    if initial_date_column_name and initial_date_column_name in df.columns: # If date column was valid, retain it using its original name
        cols_to_retain.append(initial_date_column_name)

    # Ensure filtered_df preserves original index for recommendations if possible
    # And explicitly select columns to retain, including the date_column_name if present
    # Make sure to handle cases where 'Title' or 'Description' might not be in filtered_df after slicing
    filtered_df = df.iloc[original_indices].copy()
    filtered_df = filtered_df[cols_to_retain]
    filtered_df.index = df.iloc[original_indices].index # Keep original index if available

    if not filtered_texts_clean:
        raise ValueError("No valid text descriptions found after preprocessing.")

    # 3. Extract BERT Embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(filtered_texts_clean, show_progress_bar=False) # Set to False for Gradio interface

    # 4. Reduce Dimensions with PCA
    pca = PCA(n_components=50, random_state=42)
    X_pca = pca.fit_transform(embeddings)

    # 5. Choose Optimal K Automatically
    silhouette_scores = []
    K_range = range(2, min(11, len(X_pca) - 1)) # Ensure K does not exceed number of samples - 1
    if len(K_range) < 2: # Not enough samples or K_range too small for clustering
        raise ValueError("Not enough data points or range for K to perform clustering effectively.")

    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_pca)
        if len(np.unique(labels)) > 1 and len(X_pca) > 1:
            score = silhouette_score(X_pca, labels)
            silhouette_scores.append(score)
        else:
            silhouette_scores.append(-1)

    if not silhouette_scores or all(s == -1 for s in silhouette_scores):
        best_k = K_range[0] if K_range else 2
    else:
        best_k = K_range[np.argmax(silhouette_scores)]

    # 6. Apply Final K-Means
    kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    labels_final = kmeans_final.fit_predict(X_pca)
    filtered_df['cluster'] = labels_final

    # 6.1 Define category keywords for semantic naming
    category_keywords = {
        'Sports': ['sport', 'game', 'team', 'match', 'league', 'win', 'player', 'coach', 'season', 'championship'],
        'Politics': ['government', 'election', 'president', 'party', 'vote', 'policy', 'law', 'political', 'congress', 'senate'],
        'Technology': ['tech', 'new', 'company', 'internet', 'software', 'data', 'phone', 'app', 'digital', 'ai'],
        'Business': ['market', 'company', 'price', 'shares', 'economy', 'investor', 'bank', 'business', 'financial', 'growth'],
        'Health': ['health', 'doctor', 'patient', 'medical', 'virus', 'disease', 'study', 'research', 'hospital', 'drug']
    }

    # 7. Evaluate Clustering (Silhouette Score)
    final_silhouette_score = silhouette_score(X_pca, labels_final)

    # 8. Extract Top Keywords for each cluster and assign semantic names
    keywords_output = []
    semantic_cluster_names = {}

    for i in range(best_k):
        cluster_texts = filtered_df[filtered_df['cluster']==i]['Description'].tolist()
        cluster_keywords = get_top_keywords(cluster_texts)

        best_match_category = 'General Topic ' + str(i)
        max_score = 0

        for category, keywords_list in category_keywords.items():
            score = sum(1 for keyword in keywords_list if keyword in cluster_keywords)
            if score > max_score:
                max_score = score
                best_match_category = category

        if max_score < 2:
            semantic_cluster_names[i] = 'General Topic ' + str(i)
        else:
            semantic_cluster_names[i] = best_match_category

        keywords_output.append(f"{semantic_cluster_names[i]} (Cluster {i}) Top Keywords: {', '.join(cluster_keywords)}")
    formatted_keywords_string = "\n".join(keywords_output)

    # Update cluster names in DataFrame and distribution
    filtered_df['cluster_name'] = filtered_df['cluster'].map(semantic_cluster_names)
    cluster_distribution = filtered_df['cluster_name'].value_counts().reset_index()
    cluster_distribution.columns = ['Cluster Name', 'Count']

    # 9. Get 2D PCA for plotting (now directly generate the plot)
    pca_2d = PCA(n_components=2, random_state=42)
    X_2D = pca_2d.fit_transform(embeddings)

    pca_plot_fig = plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=X_2D[:, 0], y=X_2D[:, 1],
        hue=filtered_df['cluster_name'],
        palette=sns.color_palette('deep', n_colors=best_k),
        legend='full',
        alpha=0.6
    )
    plt.title('Article Clusters (2D PCA)')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.grid(True)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.close(pca_plot_fig) # Close the figure to prevent display in non-Gradio contexts

    # 10. Generate recommendations if article_id is provided
    recommendations_df = pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])
    if article_id_rec is not None and not pd.isna(article_id_rec):
        try:
            article_id_rec_int = int(article_id_rec)
            recommendations_df = get_recommendations(article_id_rec_int, filtered_df, embeddings, n_recommendations_rec)
        except ValueError as e:
            print(f"Warning: Could not process article_id for recommendation: {e}")
        except Exception as e:
            print(f"Error generating recommendations: {e}")

    # 11. Perform Trend Analysis and generate plot
    trend_plot_fig = None
    if date_column_name and date_column_name in filtered_df.columns:
        trend_data = perform_trend_analysis(filtered_df, date_column_name)
        if trend_data is not None and not trend_data.empty:
            trend_plot_fig = plt.figure(figsize=(12, 6))
            sns.lineplot(data=trend_data, x='Date', y='count', hue='cluster_name', marker='o')
            plt.title('Article Trends by Cluster Over Time')
            plt.xlabel('Date')
            plt.ylabel('Number of Articles')
            plt.xticks(rotation=45)
            plt.grid(True)
            plt.legend(title='Cluster')
            plt.tight_layout()
            plt.close(trend_plot_fig)
        else:
            # If trend_data is empty or None, create an empty plot figure to return
            trend_plot_fig = plt.figure()
            plt.title('No Trend Data Available')
            plt.text(0.5, 0.5, 'Insufficient data to plot trend.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
            plt.axis('off')
    else:
        # If date_column_name is not provided or invalid, create an empty plot figure
        trend_plot_fig = plt.figure()
        plt.title('Date Column Not Provided or Invalid for Trend Analysis')
        plt.text(0.5, 0.5, 'Please provide a valid date column name to generate trend analysis.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
        plt.axis('off')

    # Return plot figures instead of raw dataframes and embeddings
    return best_k, final_silhouette_score, formatted_keywords_string, cluster_distribution, pca_plot_fig, recommendations_df, trend_plot_fig

print("Gradio app logic function 'cluster_news_articles' modified to return plot figures.")

Gradio app logic function 'cluster_news_articles' modified to return plot figures.


In [ ]:
import gradio as gr

# Create the Gradio interface
interface = gr.Interface(
    fn=cluster_news_articles,
    inputs=[
        gr.File(label="Upload CSV File"),
        gr.Number(label="Article ID for Recommendation (Optional)", value=0, interactive=True),
        gr.Number(label="Number of Recommendations (Optional)", value=5, interactive=True),
        gr.Textbox(label="Date Column Name (Optional, for Trend Analysis)", placeholder="e.g., publish_date", interactive=True)
    ],
    outputs=[
        gr.Number(label="Optimal Number of Clusters (k)"),
        gr.Number(label="Silhouette Score"),
        gr.Textbox(label="Top Keywords Per Cluster"),
        gr.Dataframe(label="Cluster Distribution"),
        gr.Plot(label="2D PCA Visualization"),
        gr.Dataframe(label="Recommended Articles"),
        gr.Plot(label="Trend Analysis Plot (if date column provided)") # New output for trend analysis
    ],
    title="News Article Clustering with BERT Embeddings and K-Means",
    description="Upload a CSV file containing a 'Description' column to cluster news articles, get recommendations, and analyze trends over time."
)

# Launch the interface
interface.launch(share=True)

print("Gradio interface updated with date column input and trend analysis plot output.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7dd2021c2f9993ed6d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio interface updated with date column input and trend analysis plot output.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN # Only DBSCAN needed now
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.metrics.pairwise import cosine_similarity

# Helper function for text preprocessing
def preprocess_text(text):
    text = str(text).lower() # Ensure text is string and convert to lowercase
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove non-alphabetic characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Helper function for keyword extraction
def get_top_keywords(texts, n_keywords=10):
    texts = [text for text in texts if text and text.strip() != '']
    if not texts:
        return []
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    X = vectorizer.fit_transform(texts)
    sums = X.sum(axis=0)
    words = vectorizer.get_feature_names_out()
    keywords = [words[i] for i in np.argsort(sums.A1)[-n_keywords:][::-1]]
    return keywords

# Recommendation function (re-defined to be self-contained for the new function)
def get_recommendations_dbscan(article_id, df_clustered, embeddings, n_recommendations=5):
    if article_id not in df_clustered.index:
        if pd.isna(article_id):
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])
        try:
            article_id = int(article_id)
        except ValueError:
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])

        if article_id not in df_clustered.index:
            return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])

    target_cluster_label = df_clustered.loc[article_id, 'cluster']

    # If the target article is noise, it cannot have recommendations based on its cluster
    if target_cluster_label == -1:
        print(f"Article ID {article_id} is classified as noise (cluster -1), no cluster-based recommendations.")
        return pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])

    target_embedding_index = df_clustered.index.get_loc(article_id)
    target_embedding = embeddings[target_embedding_index].reshape(1, -1)

    cluster_articles = df_clustered[df_clustered['cluster'] == target_cluster_label]
    cluster_articles = cluster_articles[cluster_articles.index != article_id]

    if cluster_articles.empty:
        return pd.DataFrame(columns=['Title', 'Description', 'Similarity'])

    cluster_article_original_indices = [df_clustered.index.get_loc(idx) for idx in cluster_articles.index]
    cluster_embeddings = embeddings[cluster_article_original_indices]

    similarities = cosine_similarity(target_embedding, cluster_embeddings).flatten()

    recommendations_df = cluster_articles.copy()
    recommendations_df['Similarity'] = similarities

    recommendations_df = recommendations_df.sort_values(by='Similarity', ascending=False)

    return recommendations_df.head(n_recommendations)[['Title', 'Description', 'cluster_name', 'Similarity']]

# Define the trend analysis function (re-defined to be self-contained) - now returns DataFrame
def perform_trend_analysis_dbscan(df_clustered, date_column_name):
    if not date_column_name or date_column_name not in df_clustered.columns:
        return None

    df_clustered_copy = df_clustered.copy()
    df_clustered_copy[date_column_name] = pd.to_datetime(df_clustered_copy[date_column_name])

    trend_data = df_clustered_copy.groupby([df_clustered_copy[date_column_name].dt.date, 'cluster_name']).size().reset_index(name='count')
    trend_data = trend_data.rename(columns={date_column_name: 'Date'})

    if trend_data.empty:
        return pd.DataFrame(columns=['Date', 'cluster_name', 'count'])

    return trend_data

# Define the main clustering function for Gradio using DBSCAN - updated to return plot figures
def cluster_news_articles_dbscan(file_obj, article_id_rec=None, n_recommendations_rec=5, date_column_name=None, eps=0.5, min_samples=5):
    # 1. Load Data
    df = pd.read_csv(file_obj.name)
    if 'Description' not in df.columns:
        raise ValueError("CSV file must contain a 'Description' column.")

    # 1.1 Process date column if provided
    initial_date_column_name = date_column_name # Store original name
    if date_column_name and date_column_name in df.columns:
        df[date_column_name] = pd.to_datetime(df[date_column_name], errors='coerce')
        df.dropna(subset=[date_column_name], inplace=True)
    else:
        date_column_name = None

    texts = df['Description'].astype(str)

    # 2. Preprocess Text
    texts_clean = texts.apply(preprocess_text).tolist()
    original_indices = [i for i, text in enumerate(texts_clean) if text and text.strip() != '']
    filtered_texts_clean = [texts_clean[i] for i in original_indices]

    cols_to_retain = ['Title', 'Description']
    if initial_date_column_name and initial_date_column_name in df.columns:
        cols_to_retain.append(initial_date_column_name)

    filtered_df = df.iloc[original_indices].copy()
    filtered_df = filtered_df[cols_to_retain]
    filtered_df.index = df.iloc[original_indices].index

    if not filtered_texts_clean:
        raise ValueError("No valid text descriptions found after preprocessing.")

    # 3. Extract BERT Embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(filtered_texts_clean, show_progress_bar=False)

    # 4. Reduce Dimensions with PCA
    pca = PCA(n_components=50, random_state=42)
    X_pca = pca.fit_transform(embeddings)

    # 5. Apply DBSCAN
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels_final = dbscan.fit_predict(X_pca)

    unique_clusters = np.unique(labels_final)
    n_clusters_found = len(unique_clusters[unique_clusters != -1])

    # 6. Evaluate Clustering (Silhouette Score) - only for non-noise points
    final_silhouette_score = -1
    if n_clusters_found > 1 and len(X_pca[labels_final != -1]) > 1:
        X_pca_filtered = X_pca[labels_final != -1]
        labels_final_filtered = labels_final[labels_final != -1]
        if len(np.unique(labels_final_filtered)) > 1:
             final_silhouette_score = silhouette_score(X_pca_filtered, labels_final_filtered)
        else:
            print("Warning: Only one cluster found after filtering noise for silhouette score.")
    elif n_clusters_found == 1 and len(X_pca[labels_final != -1]) > 1:
         print("Warning: Only one cluster found by DBSCAN (excluding noise). Silhouette score cannot be calculated.")
    else:
        print("Warning: Not enough data points or clusters for silhouette score calculation after DBSCAN.")

    filtered_df['cluster'] = labels_final

    # 6.1 Define category keywords for semantic naming
    category_keywords = {
        'Sports': ['sport', 'game', 'team', 'match', 'league', 'win', 'player', 'coach', 'season', 'championship'],
        'Politics': ['government', 'election', 'president', 'party', 'vote', 'policy', 'law', 'political', 'congress', 'senate'],
        'Technology': ['tech', 'new', 'company', 'internet', 'software', 'data', 'phone', 'app', 'digital', 'ai'],
        'Business': ['market', 'company', 'price', 'shares', 'economy', 'investor', 'bank', 'business', 'financial', 'growth'],
        'Health': ['health', 'doctor', 'patient', 'medical', 'virus', 'disease', 'study', 'research', 'hospital', 'drug']
    }

    # 8. Extract Top Keywords for each cluster and assign semantic names
    keywords_output = []
    semantic_cluster_names = {}

    for i in unique_clusters:
        if i == -1:
            semantic_cluster_names[i] = 'Noise'
            noise_texts = filtered_df[filtered_df['cluster'] == i]['Description'].tolist()
            noise_keywords = get_top_keywords(noise_texts)
            if noise_keywords:
                keywords_output.append(f"Noise (Cluster {i}) Top Keywords: {', '.join(noise_keywords)}")
            else:
                keywords_output.append(f"Noise (Cluster {i}) No keywords found.")
            continue

        cluster_texts = filtered_df[filtered_df['cluster'] == i]['Description'].tolist()
        cluster_keywords = get_top_keywords(cluster_texts)

        best_match_category = 'General Topic ' + str(i)
        max_score = 0

        for category, keywords_list in category_keywords.items():
            score = sum(1 for keyword in keywords_list if keyword in cluster_keywords)
            if score > max_score:
                max_score = score
                best_match_category = category

        if max_score < 2:
            semantic_cluster_names[i] = 'General Topic ' + str(i)
        else:
            semantic_cluster_names[i] = best_match_category

        keywords_output.append(f"{semantic_cluster_names[i]} (Cluster {i}) Top Keywords: {', '.join(cluster_keywords)}")
    formatted_keywords_string = "\n".join(keywords_output)

    filtered_df['cluster_name'] = filtered_df['cluster'].map(semantic_cluster_names)
    cluster_distribution = filtered_df['cluster_name'].value_counts().reset_index()
    cluster_distribution.columns = ['Cluster Name', 'Count']

    # 9. Get 2D PCA for plotting
    pca_2d = PCA(n_components=2, random_state=42)
    X_2D = pca_2d.fit_transform(embeddings)

    # Generate PCA plot figure
    pca_plot_fig = plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=X_2D[:, 0], y=X_2D[:, 1],
        hue=filtered_df['cluster_name'],
        palette=sns.color_palette('deep', n_colors=len(unique_clusters)),
        legend='full',
        alpha=0.6
    )
    plt.title('Article Clusters (2D PCA)')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.grid(True)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.close(pca_plot_fig)

    # 10. Generate recommendations if article_id is provided
    recommendations_df = pd.DataFrame(columns=['Title', 'Description', 'cluster_name', 'Similarity'])
    if article_id_rec is not None and not pd.isna(article_id_rec):
        try:
            article_id_rec_int = int(article_id_rec)
            recommendations_df = get_recommendations_dbscan(article_id_rec_int, filtered_df, embeddings, n_recommendations_rec)
        except ValueError as e:
            print(f"Warning: Could not process article_id for recommendation: {e}")
        except Exception as e:
            print(f"Error generating recommendations: {e}")

    # 11. Perform Trend Analysis - now returns DataFrame and generates plot
    trend_data = None
    trend_plot_fig = None
    if date_column_name and date_column_name in filtered_df.columns:
        trend_data = perform_trend_analysis_dbscan(filtered_df, date_column_name)
        if trend_data is not None and not trend_data.empty:
            trend_plot_fig = plt.figure(figsize=(12, 6))
            sns.lineplot(data=trend_data, x='Date', y='count', hue='cluster_name', marker='o')
            plt.title('Article Trends by Cluster Over Time')
            plt.xlabel('Date')
            plt.ylabel('Number of Articles')
            plt.xticks(rotation=45)
            plt.grid(True)
            plt.legend(title='Cluster')
            plt.tight_layout()
            plt.close(trend_plot_fig)
        else:
            # If trend_data is empty or None, create an empty plot figure to return
            trend_plot_fig = plt.figure()
            plt.title('No Trend Data Available')
            plt.text(0.5, 0.5, 'Insufficient data to plot trend.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
            plt.axis('off')
    else:
        # If date_column_name is not provided or invalid, create an empty plot figure
        trend_plot_fig = plt.figure()
        plt.title('Date Column Not Provided or Invalid for Trend Analysis')
        plt.text(0.5, 0.5, 'Please provide a valid date column name to generate trend analysis.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
        plt.axis('off')

    # Return plot figures instead of raw dataframes and embeddings, and the recommendations_df
    return n_clusters_found, final_silhouette_score, formatted_keywords_string, cluster_distribution, pca_plot_fig, recommendations_df, trend_plot_fig

print("DBSCAN clustering function 'cluster_news_articles_dbscan' modified to return plot figures.")

DBSCAN clustering function 'cluster_news_articles_dbscan' modified to return plot figures.


In [ ]:
import gradio as gr

# Create the Gradio interface for DBSCAN
interface_dbscan = gr.Interface(
    fn=cluster_news_articles_dbscan,
    inputs=[
        gr.File(label="Upload CSV File"),
        gr.Number(label="Article ID for Recommendation (Optional)", value=0, interactive=True),
        gr.Number(label="Number of Recommendations (Optional)", value=5, interactive=True),
        gr.Textbox(label="Date Column Name (Optional, for Trend Analysis)", placeholder="e.g., publish_date", interactive=True),
        gr.Number(label="DBSCAN Epsilon (eps)", value=0.5, interactive=True, info="Maximum distance between two samples for one to be considered as in the neighborhood of the other."),
        gr.Number(label="DBSCAN Min Samples", value=5, interactive=True, info="The number of samples (or total weight) in a neighborhood for a point to be considered as a core point.")
    ],
    outputs=[
        gr.Number(label="Number of Clusters Found"),
        gr.Number(label="Silhouette Score"),
        gr.Textbox(label="Top Keywords Per Cluster"),
        gr.Dataframe(label="Cluster Distribution"),
        gr.Plot(label="2D PCA Visualization"),
        gr.Dataframe(label="Recommended Articles"),
        gr.Plot(label="Trend Analysis Plot (if date column provided)")
    ],
    title="News Article Clustering with BERT Embeddings and DBSCAN",
    description="Upload a CSV file containing a 'Description' column to cluster news articles using DBSCAN, get recommendations, and analyze trends over time. Use 'eps' and 'min_samples' to control DBSCAN behavior."
)

# Launch the new DBSCAN interface
interface_dbscan.launch(share=True)

print("New Gradio interface for DBSCAN launched.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f130b09d5742cd3aad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


New Gradio interface for DBSCAN launched.
